# S12 · One messy dataset → one working model

We take a single deliberately ugly customer table and run it through the whole
feature-engineering job: a baseline model, cleaning the mess, and then every
transform from the session — missing values, outliers, log, dates, interaction,
polynomial, selection, leakage, one-hot and scaling — and watch the score climb.

One dataset. One model family. The only thing that changes is how we write the
columns down.

## Setup

This notebook uses numpy, pandas and scikit-learn.
Google Colab already ships all three, so there is nothing to install.

In [ ]:
# fast maths on whole columns of numbers
import numpy as np
import pandas as pd

# a fixed seed so we all get the same 'random' mess every time
np.random.seed(42)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

print("Setup complete - nothing to install.")

## Step 1 — build the deliberately messy dataset

Every classic data-quality problem, in one table:

* a **skewed** money column (`Income`) with **missing** values and **outliers**,
* a **correlated twin** (`Income_Copy`), and a **leakage** column (`Purchased_Copy`),
* a **constant** column (`Region`), a **unique ID** (`Customer_ID`),
* a text column with **fake-missing** `999` and **untidy casing** (`City`),
* a **date** column, and a few **duplicate rows**.

We build it in code, save it to `data/messy_customers.csv`, and pretend a colleague
just handed us the file.

In [ ]:
# The recipe for the mess — so everyone sees exactly what is in it.
n = 500

df = pd.DataFrame({
    "Customer_ID": ["CUST-%04d" % i for i in range(1, n + 1)],   # unique identifier
    "Age":         np.random.randint(18, 70, n),
    "Income":      np.random.exponential(50000, n),              # skewed money
    "Income_Copy": np.random.exponential(50000, n),              # will become a twin
    "City":        np.random.choice(["Pune", "Mumbai", "Delhi", "Bangalore"], n),
    "Join_Date":   pd.date_range("2022-01-01", periods=n, freq="D"),
})

# make the twin perfectly correlated with Income
df["Income_Copy"] = df["Income"] * 1.02

# the answer: bought when older AND richer
df["Purchased"] = ((df["Age"] > 40) & (df["Income"] > 40000)).astype(int)

# missing values (the twin misses the same rows, so it stays a true twin)
missing_idx = np.random.choice(n, 50)
df.loc[missing_idx, "Income"] = np.nan
df.loc[missing_idx, "Income_Copy"] = np.nan
missing_idx = np.random.choice(n, 30)
df.loc[missing_idx, "City"] = np.nan

# outliers
df.loc[0, "Income"] = 1000000
df.loc[1, "Income"] = 1500000
df.loc[0, "Income_Copy"] = 1020000
df.loc[1, "Income_Copy"] = 1530000

# fake-missing 999 hiding in Age
fake_idx = np.random.choice(n, 15)
df.loc[fake_idx, "Age"] = 999

# untidy city text (spaces + lower case)
messy_idx = np.random.choice(n, 15)
df.loc[messy_idx, "City"] = df.loc[messy_idx, "City"].apply(
    lambda c: " " + c.lower() + " ")

# constant column + a leakage column
df["Region"] = "West"
df["Purchased_Copy"] = df["Purchased"]

# a few exact duplicate rows
df = pd.concat([df, df.iloc[[3, 42, 117]].copy()], ignore_index=True)

# save a copy, so students can also load the file directly
df.to_csv("data/messy_customers.csv", index=False)

print("rows:", len(df))

## Step 2 — always look first

Before touching anything, read the file back the way the bank gave it to us and
profile it: how many rows, what types, how much is missing, what looks wrong.

In [ ]:
df = pd.read_csv("data/messy_customers.csv")

print("shape:", df.shape)
print(df.head())
print()
print(df.info())
print()
print("missing values per column:")
print(df.isna().sum())
print()
print("largest age (a 999 is a disguised missing value):", df["Age"].max())
print("duplicate rows:", df.duplicated().sum())

## Step 3 — baseline model on the raw mess

Train a plain logistic regression on the simplest usable columns and measure where
we start. This number is our 'before'.

In [ ]:
# keep only the columns a model can almost use right away
temp = df[["Age", "Income", "Purchased"]].copy()

# quick-and-dirty fill so the model can run at all
temp["Income"] = temp["Income"].fillna(temp["Income"].median())

X = temp.drop("Purchased", axis=1)
y = temp["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, pred), 4))

## Step 4 — handle missing values

Unmask the disguises first (999 is not a 999-year-old customer), tidy the text so
`Mumbai`, ` mumbai ` and `MUMBAI` count as one city, then fill the blanks with a
robust median / most-common value.

In [ ]:
# unmask the fake-missing 999 in Age
df["Age"] = df["Age"].replace(999, np.nan)

# tidy the city text BEFORE counting categories
df["City"] = df["City"].str.strip().str.title()

print("missing after unmasking:")
print(df.isna().sum())

# fill numeric gaps with the median, text gaps with the most common value
df["Income"] = df["Income"].fillna(df["Income"].median())
df["City"] = df["City"].fillna(df["City"].mode()[0])
df["Age"] = df["Age"].fillna(df["Age"].median())

print("\nmissing after filling:")
print(df.isna().sum())

## Step 5 — drop the dead weight

Remove rows and columns that can never carry signal: exact duplicate rows, the
constant column (every value the same) and the unique customer ID (an identifier,
not a feature).

In [ ]:
# remove exact duplicate rows
df = df.drop_duplicates()

# drop the constant column and the unique identifier
df = df.drop(columns=["Region", "Customer_ID"])

print("rows after cleaning:", len(df))
print("columns now:", list(df.columns))

## Step 6 — handle outliers

Two incomes are ten times the rest. Cap them at a sensible limit using the
inter-quartile range so a couple of rows cannot bend the whole model.

In [ ]:
q1 = df["Income"].quantile(0.25)
q3 = df["Income"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("income cap:", round(upper, 0))
print("largest income BEFORE:", round(df["Income"].max(), 0))

df["Income"] = np.clip(df["Income"], lower, upper)
df["Income_Copy"] = np.clip(df["Income_Copy"], lower, upper)

print("largest income AFTER capping:", round(df["Income"].max(), 0))

## Step 7 — log transform

Income is skewed (long right tail). A `log` pulls the big values back so the model
sees a roughly symmetric column.

In [ ]:
df["Income_Log"] = np.log1p(df["Income"])

print(df[["Income", "Income_Log"]].head())

## Step 8 — date features

One date unfolds into many numbers: month, weekday, and so on.

In [ ]:
# make sure it really is a date column
df["Join_Date"] = pd.to_datetime(df["Join_Date"])

df["Month"]   = df["Join_Date"].dt.month
df["Weekday"] = df["Join_Date"].dt.dayofweek

print(df[["Join_Date", "Month", "Weekday"]].head())

## Step 9 — interaction feature

A combination column, `Age × Income`, lets the model see the *joint* signal that
neither column holds alone.

In [ ]:
df["Age_Income"] = df["Age"] * df["Income_Log"]

print(df[["Age", "Income_Log", "Age_Income"]].head())

## Step 10 — polynomial feature

Give the model `x` and `x²`, so a linear model can bend.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[["Age"]])

print("generated columns:", poly.get_feature_names_out(["Age"]))

df["Age_Squared"] = poly_features[:, 1]

## Step 11 — feature selection: drop the correlated twin

`Income_Copy` moves in lockstep with `Income` — it carries the same information
twice. Keep one, drop the other.

In [ ]:
print(df[["Income", "Income_Copy"]].corr())

df = df.drop(columns="Income_Copy")

print("\nDropped Income_Copy because it is highly correlated with Income.")

## Step 12 — the leakage trap

`Purchased_Copy` is identical to the target. If we leave it in, the model looks
perfect — and would fail in production, because the real answer is not in the
real data. This is cheating. Drop it.

In [ ]:
leak = df[["Age", "Income_Log", "Purchased_Copy"]].copy()
ly = df["Purchased"]
LX_train, LX_test, ly_train, ly_test = train_test_split(
    leak, ly, test_size=0.3, random_state=42)

leak_model = LogisticRegression().fit(LX_train, ly_train)
print("accuracy WITH the leakage column:",
      round(accuracy_score(ly_test, leak_model.predict(LX_test)), 4))

df = df.drop(columns="Purchased_Copy")
print("Dropped Purchased_Copy — the answer must never leak into the features.")

## Step 13 — one-hot encoding

Turn the tidy city column into 0/1 columns, no fake order. `drop_first=True` keeps
the number of columns down (the dropped city is implied by all zeros).

In [ ]:
df = pd.get_dummies(df, columns=["City"], drop_first=True)

print(df.head())

## Step 14 — scaling (fit on TRAIN only)

Now build the full engineered feature matrix and put the numeric columns on one
footing. The golden rule: the scaler learns its numbers from the **training rows
only**, then transforms both splits — otherwise the test set leaks into the fit.

In [ ]:
drop_cols = ["Purchased", "Join_Date", "Income"]
X = df.drop(columns=drop_cols)
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

numeric_features = ["Age", "Income_Log", "Age_Income", "Age_Squared"]

scaler = StandardScaler()
scaler.fit(X_train[numeric_features])          # learn from TRAIN only
X_train[numeric_features] = scaler.transform(X_train[numeric_features])
X_test[numeric_features]  = scaler.transform(X_test[numeric_features])

## Step 15 — the final model

Same logistic regression, same split — but now the columns are engineered.
Compare against the baseline from Step 3.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Baseline Accuracy:", round(0.7152, 4))
print("Final Accuracy:   ", round(accuracy_score(y_test, pred), 4))

## What you just did

Same model family, same rows — only the columns changed:

* missing → unmasked and filled,
* dead weight (duplicates, constant, ID) → dropped,
* outliers → capped, skewed income → logged,
* dates, interactions and polynomials → engineered,
* correlated twin and the leakage column → removed,
* categories → one-hot, numbers → scaled on train only.

We didn't change the model. We changed the representation of the data — and the
score climbed from the baseline to the final model.